2019년 10월 ~ 2020년 2월 까지의 월별 데이터 테이블 <br>
(그러므로 모든 테이블 컬럼 구성이 동일함)

평균적으로 400만개의 행이 있음

category_code(전체 행 대비 90% 이상이 결측치) <br>brand(약 30%가 결측치) <br>user_session(약 1000행 미만이 결측치) 컬럼에서 <br>꾸준히 결측치가 발생하는 패턴이 보임 (이외에는 결측치 없음)

브랜드 runnail / irisk / masura 가 1,2,3위를 다투고 있음


# user_id vs user_session 핵심 차이 정리

데이터 분석 시 분석 단위를 '사람'으로 잡을지, '방문'으로 잡을지에 따라 활용 컬럼이 달라집니다. <br>

---

## 1. user_id (사용자 고유 식별자)
* **정의**: 서비스에 가입하거나 접속한 **개별 사용자**에게 부여된 고유 번호입니다. <br>
* **특징**: 사용자가 로그아웃했다가 다시 들어와도, 기기를 바꿔도(로그인 시) 변하지 않는 고유값입니다. <br>

## 2. user_session (방문 단위 식별자)
* **정의**: 사용자가 한 번 접속해서 **활동을 마치고 떠날 때까지의 일련의 과정**에 부여된 임시 번호입니다. <br>
* **특징**: 동일한 `user_id`를 가진 사람이라도 아침에 접속했을 때와 저녁에 접속했을 때의 `user_session` 값은 다릅니다. (보통 30분간 활동이 없으면 세션이 종료된 것으로 간주합니다.) <br>

---

## 비교

| 구분 | user_id | user_session |
| :--- | :--- | :--- |
| **관점** | 사람 (Person) | 방문/세션 (Visit) |
| **지속성** | 영구적 (변경 안 됨) | 일시적 (접속 시마다 생성) |
| **관계** | 1개 (부모) | N개 (자식 - 한 명의 유저는 여러 세션을 가짐) |

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
from sklearn.linear_model import LinearRegression
import time

In [ ]:
# 한글 안깨지게, 특수문자 안깨지게, 한글 선명하게
plt.rc("font", family="Malgun Gothic")
plt.rc("axes", unicode_minus=False)

# 폰트가 선명하게 보이도록 retina 설정
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats("retina")

# 한글폰트와 마이너스 폰트 설정 확인
pd.Series([-1, 0, 1, 3, 5]).plot(title="한글폰트")

# 지수 마이너스 유니코드 안깨지게
import matplotlib as mpl
mpl.rcParams['axes.unicode_minus'] = False

# 경고메시지 스킵
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 데이터 불러오기

oct_2019 = pd.read_csv("2019-Oct.csv")
nov_2019 = pd.read_csv("2019-Nov.csv")
dec_2019 = pd.read_csv("2019-Dec.csv")
jan_2020 = pd.read_csv("2020-Jan.csv")
feb_2020 = pd.read_csv("2020-Feb.csv")

In [ ]:
# Union 실시
df = pd.concat([oct_2019, nov_2019, dec_2019, jan_2020, feb_2020], axis=0, ignore_index=True, join='inner')

In [ ]:
# 전체 2,000만 행

'''
컬럼 구성 unique값
1. event_type           Distinct: 4
view                    47%
cart                    28%
remove_from_cart        19%
purchase                6%      -> 얘만 봐야지

2. product_id           Distinct: 54571
3. category_id          Distinct: 525
4. category_code        Distinct: 12
5. brand                Distinct: 273
6. price                Distinct: 2860
7. user_id              Distinct: 1639358
8. user_session         Distinct: 4535941
'''

display(df.info(),
        df,
        df.isna().sum())

In [ ]:
# 카테고리 코드 구성 확인
# 화장품 관련된 제품은 cosmetic_bag(파우치) 뿐임
# air_coditioner 가 헤어 컨디셔너 오타일수도 있겠다 생각했으나 appliances가 가전제품인것으로 미뤄보았을떄 그럴수가없음

df['category_code'].unique()

In [ ]:
# -가격인 행들은 모두 카테고리 코드, 브랜드가 결측치이고 카테고리id가 동일함
# 데이터 제공자의 Q&A 확인결과, 단순 오류라고 함.

'''
event_type은
purchase            96%
remove_from_cart    2%
view                2%
'''

minus_price = df[df['price'] < 0]
minus_price

# 필터링

df = df[df['price'] > 0]

In [ ]:
# 코스메틱 시장 실 매출 순위 파악

real_sales = df[df['event_type'] == 'purchase']
real_sales.groupby('brand')['price'].sum()

In [ ]:
# 네일머신 관련 브랜드 중 객단가가 과도하게 높으면 네일머신 등을 주력상품으로 하고 있다는것을 알 수 있음
# 네일머신을 전문적으로 판매하는 브랜드를 제외하고 네일아트 제품을 판매하는 브랜드는 객단가

brand_sales_mean = df.groupby('brand')['price'].mean()
brand_sales_mean

In [ ]:
# .tolist()를 사용하여 파이썬 리스트로 변환 후 브랜드명 끊김없이 출력
print(df['brand'].unique().tolist())

## Q. 카테고리 코드 컬럼에서 가전에 램프 등 있을수도 있지 않나?

In [ ]:
# 카테고리 컬럼을 그냥 드랍하기 전에 한번 더 체크해보자
df_full = pd.concat([oct_2019, nov_2019, dec_2019, jan_2020, feb_2020], axis=0, ignore_index=True, join='inner')

In [ ]:
# 카테고리코드가 있는 행과 목표 브랜드를 비교해보면 힌트를 얻을 수 있을것
# 4개의 대표적인 분류를 확인 가능

df_catna_ru = df_full[(df_full['category_code'].notna())&(df_full['brand'] == 'runail')]
df_catna_ru

In [ ]:
# 카테고리코드가 있는 행과 목표 브랜드를 비교해보면 힌트를 얻을 수 있을것
# 3개의 대표적인 분류를 확인 가능

df_catna_ir = df_full[(df_full['category_code'].notna())&(df_full['brand'] == 'irisk')]
df_catna_ir

In [ ]:
# 특이하게도 그라쏠의 경우에는 해당사항이 없었음

df_catna_gr = df_full[(df_full['category_code'].notna())&(df_full['brand'] == 'grattol')]
df_catna_gr

1. 가전 분야 (appliances)
appliances.environment.vacuum : 네일 샵에서 손톱을 갈 때 발생하는 가루를 빨아들이는 '네일 흡진기'

2. 가구 분야 (furniture)
furniture.bathroom.bath : 페디큐어 를 할 때 사용하는 '족욕기'나 '발 대야'가 욕조로 분류되었을 수 있습니다. <br>
furniture.living_room.cabinet : 수만 개의 젤 네일 병을 보관하는 '네일 진열장'이나 '매니큐어 렉'이 일반 가구인 캐비닛으로 인식된 것

3. 기타 액세서리
accessories.bag (파우치)

In [ ]:
# 네일아트 브랜드 리스트
# 필터링 후 데이터 개수: 7526092 인데, 브랜드가 없는게 870만개였던걸 고려하면, 타 도메인 데이터를 다 합쳐도 500만개인것을 알 수 있음



target_brands = [
    'runail', 'lianail', 'irisk', 'masura', 'ingarden', 'de.lux', 'milv', 
    'f.o.x', 'cnd', 'grattol', 'bluesky', 'pole', 'jessnail', 'haruyama', 
    'rosi', 'airnails', 'beautix', 'uno', 'entity', 'beauty-free', 
    'kinetics', 'cosmoprofi', 'pnb', 'domix', 'enas', 'oniq', 'sophin', 
    'uskusi', 'solomeya', 'artex', 'orly', 'tertio', 'inm', 'candy', 
    'i-laq', 'blixz', 'lamixx', 'opi', 'enigma', 'rocknailstar', 
    'vl-gel', 'pueen', 'naomi', 'ibd', 'dartnails', 'yllozure'
]

# brand 컬럼의 값이 target_brands 리스트에 포함된 행만 추출
df_nail = df[df['brand'].isin(target_brands)].copy()

# 결과 확인
print(df_nail['brand'].unique())
print(f"필터링 후 데이터 개수: {len(df_nail)}")

In [ ]:
# 전처리 한방에 끝내기 (빅데이터다 보니, 태블로 Prep 활용하여 대시보드에 월별로 붙이기 위함)

# 데이터프레임 한 변수에 담기
data_frames = [oct_2019, nov_2019, dec_2019, jan_2020, feb_2020]

# 위에서 확인해 본 결과, 4가지 제품분류가 네일아트 관련하여 사용되는것을 확인했었음. 브랜드로만 필터링하는것이 좋아보임
# 결측,오염이 심한 category_code 컬럼 drop
data_frames = [
    df[
        (df['brand'].isin(target_brands))
    ].drop(columns=['category_code'])
    for df in data_frames
]

oct2_2019, nov2_2019, dec2_2019, jan2_2020, feb2_2020 = data_frames

In [ ]:
# runail 관련 행 수 파악 / 150만

runail = df[df['brand'] == 'runail'].copy()
runail

In [ ]:
# grattol 관련 행 수 파악 / 85만

grattol = df[df['brand'] == 'grattol'].copy()
grattol

In [ ]:
# 전처리 데이터 concat
df = pd.concat([oct2_2019, nov2_2019, dec2_2019, jan2_2020, feb2_2020], axis=0, ignore_index=True, join='inner')

# - 값 제거 (마이너스 값은 eda1에서 제거 가능한지 확인완료)
df_v2 = df[df['price'] > 0].copy()

타임스탬프 UTC삭제하고 3시간 추가해서 수정필요

runail, 보로네시, 보로실로바 거리, 1/3. / UTC + 3

grattol, 니즈니노브고로드, 성. Maslyakova, 5, 112 / UTC + 3

In [ ]:
# 타임존 설정하여 시간대 변경 및 날짜컬럼으로 변환 (이렇게 해야 UTC기준이라는 전제 반영가능)
# 시행착오가 있었음

'''
event_time에 문자열과 datetime이 섞여 있었는데, .str.replace() 같은 문자열 연산을 먼저 해서 datetime 값이 NaN으로 변함
혹은 UTC가 붙은 행과 안 붙은 행이 섞여 있는데, 포맷 없이 파싱하다가 일부만 실패 → NaT
그래서 지금처럼 astype(str)로 통일 → UTC 제거 → 명시 포맷으로 파싱하니 다행히 결측 없어짐
'''

s = df_v2["event_time"].astype(str).str.replace(r"\s*UTC$", "", regex=True)

df_v2["event_time"] = pd.to_datetime(
    s,
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce",
    utc=True
).dt.tz_convert("Etc/GMT-3")

In [ ]:
# 2020-02 데이터까지만 남기기 (3시간이 더해지면서 3월 자투리데이터가 생겼음, 시각화에 악영향을 주니 제거)
cutoff = pd.Timestamp('2020-02-29 23:59:59', tz=df['event_time'].dt.tz)
df = df[df['event_time'] <= cutoff].copy()

In [ ]:
display(df_v2.info(), df_v2.head(), df_v2.isna().sum())

In [ ]:
# 전처리한 데이터 내보내기
# df_v2.to_csv('nail_market.csv', index=False)

In [ ]:
# 태블로 대시보드 제작을 위한 데이터셋 가공
df1 = pd.read_csv('nail_market.csv')
df2 = pd.read_csv('nail_market_v3.csv')

# 복합 키로 정밀하게 merge
keys = [
    "event_time","event_type","product_id","category_id","brand",
    "price","user_id","user_session"
]

df_merged = df.merge(
    df2[keys + ["cluster_tag"]],
    on=keys,
    how="left"
)

# 많은 용량을 차지하는 user_session, 효용을 다했으니 제거
df_merged_no_session = df_merged.drop(columns=["user_session"])

# 상위 10개 브랜드만 남기기
brands = ["runail","grattol","irisk","uno","masura","jessnail","ingarden","cnd","beautix","cosmoprofi"]
df_filtered = df_merged_no_session[df_merged_no_session["brand"].isin(brands)]

In [ ]:
# 풀버전 내보내기
# df_filtered.to_csv('final_full.csv', index=False)

In [ ]:
'''
# 월별로 쪼갠 버전 내보내기

df_filtered = df_filtered.copy()
df_filtered["event_time"] = pd.to_datetime(
    df_filtered["event_time"],
    format="mixed",   # 포맷이 섞여있을 때
    errors="coerce"   # 변환 실패는 NaT
)

months = ["2019-10","2019-11","2019-12","2020-01","2020-02"]
for m in months:
    ym = pd.to_datetime(m)
    out = df_filtered[
        (df_filtered["event_time"].dt.year == ym.year) &
        (df_filtered["event_time"].dt.month == ym.month)
    ]
    out.to_csv(f"df_filtered_{m}.csv", index=False)
'''

In [ ]:
df = pd.read_csv('nail_market_v3.csv')

In [ ]:
# utc 날짜컬럼으로 변환 (이렇게 해야 UTC기준이라는 전제 반영가능)
df['event_time'] = pd.to_datetime(df['event_time'], utc=True, format='%Y-%m-%d %H:%M:%S%z')
df['event_time'] = df['event_time'].dt.tz_convert('Europe/Moscow')  # +03:00 zone

# 기본 전처리 + 메모리 절약
df['event_date'] = df['event_time'].dt.date
df['event_hour'] = df['event_time'].dt.hour
df['event_type'] = df['event_type'].astype('category')
df['brand'] = df['brand'].astype('category')

In [ ]:
# 퍼널/세션 분석용 클린 데이터
df_sess = df[df['user_session'].notna()].copy()

In [ ]:
display(df.info(),
        df,
        df.isna().sum())